# FIAP — Tech Challenge Fase 2
## 03.2 — Gold Municípios

### Responsabilidade do notebook

Este notebook cria a camada **Gold Municipal**, integrando as bases Silver de:

- Municípios;
- Metas por Município.

A camada Gold transforma os dados padronizados da Silver em datasets analíticos prontos para consumo por:

- Power BI;
- análises executivas;
- modelos de Machine Learning;
- clusters de vulnerabilidade educacional.

---

### Entrada

```text
silver/municipios/ano=YYYY/TS_MUNICIPIO_YYYY.csv
silver/metas_municipios/ano=YYYY/TS_METAS_MUNICIPIOS_YYYY_SILVER.csv
```

### Saída

```text
gold/municipios/ano=YYYY/gold_municipios_YYYY.csv
gold/ranking_municipios/ranking_municipios.csv
```

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Leitura das bases

In [0]:
metadata = pd.read_parquet(CONFIG_PATH / "gold_metadata")

bases = []

for ano in [2023, 2024, 2025]:
    meta_mun = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    meta_metas = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "metas_municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    df_mun = ler_csv(
        Path(meta_mun["silver_path"]) / meta_mun["silver_file_name"]
    )

    df_metas = ler_csv(
        Path(meta_metas["silver_path"]) / meta_metas["silver_file_name"]
    )

    df_alunos = ler_csv(
        GOLD_PATH / "alunos" / f"ano={ano}" / f"GOLD_ALUNOS_{ano}.csv"
    )

    df_mun["ANO"] = ano
    df_metas["ANO"] = ano
    df_alunos["ANO"] = ano

    for df in [df_mun, df_metas, df_alunos]:
        df["CO_MUNICIPIO"] = normalizar_codigo(df["CO_MUNICIPIO"])
        df["CO_UF"] = normalizar_codigo(df["CO_UF"])

    df_integrado = (
        df_mun
        .merge(
            df_metas.drop(columns=["SG_UF", "NO_MUNICIPIO"], errors="ignore"),
            on=["ANO", "CO_UF", "CO_MUNICIPIO"],
            how="left"
        )
        .merge(
            df_alunos.drop(columns=["SG_UF", "NO_MUNICIPIO"], errors="ignore"),
            on=["ANO", "CO_UF", "CO_MUNICIPIO"],
            how="left"
        )
    )

    bases.append(df_integrado)

df_gold_municipios = pd.concat(bases, ignore_index=True)

## 5. Indicadores e risco

In [0]:
for coluna in ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP", "META_FINAL_2030"]:
    if coluna in df_gold_municipios.columns:
        df_gold_municipios[coluna] = converter_numero(df_gold_municipios[coluna])

df_gold_municipios["gap_meta_2030"] = (
    df_gold_municipios["META_FINAL_2030"]
    - df_gold_municipios["PC_ALUNO_ALFABETIZADO"]
)

df_gold_municipios["risco_educacional"] = pd.cut(
    df_gold_municipios["PC_ALUNO_ALFABETIZADO"],
    bins=[-np.inf, 50, 70, 85, np.inf],
    labels=["Crítico", "Alto", "Médio", "Baixo"]
)

df_gold_municipios["_gold_processed_at"] = datetime.now().isoformat()

if df_gold_municipios.empty:
    print("Gold Municípios sem registros.")
else:
    display(df_gold_municipios.head())

## 6. Persistência

In [0]:
for ano in [2023, 2024, 2025]:
    registro = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    salvar_csv(
        df_gold_municipios[df_gold_municipios["ANO"] == ano],
        Path(registro["gold_output_path"]),
        registro["gold_file_name"]
    )

print("Gold Municípios salva com sucesso.")